<a href="https://colab.research.google.com/github/leonardoferreiraads/1TIAPR-2026/blob/main/C%C3%B3pia_de_RN07_mlp_pratica_exercicio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sua primeira rede neural: prever se o visitante vai comprar

**Aula prática | Multilayer Perceptron (MLP) | FIAP**

## Como será a aula

1. **Agora: prática.** Vamos fazer uma rede neural funcionar, **um passo de cada vez**.
2. **Depois: teoria.** Com o resultado na tela, vamos dar nome a cada peça (neurônio, camada oculta, ativação, pesos, épocas, erro).

## O cenário

Uma **loja virtual** quer saber, para cada visitante, **se ele vai comprar ou não**. Para cada visita, temos duas informações:

| Coluna | O que é |
|---|---|
| `tempo_no_site` | minutos que o visitante ficou no site |
| `desconto` | % de desconto do cupom que ele recebeu |
| `comprou` | **1** se comprou, **0** se não comprou (é o que queremos prever) |

> Os dados desta aula foram **gerados por computador** para fins didáticos. Eles não vêm de uma loja real.

> Regra da aula: **não vamos escrever funções nossas**. Só chamamos funções prontas.

## Etapa 1: importar as ferramentas

Importar é "abrir a caixa de ferramentas". Cada linha traz uma ferramenta pronta.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split     # separar treino e teste
from sklearn.preprocessing import StandardScaler         # colocar as colunas na mesma escala
from sklearn.linear_model import LogisticRegression      # modelo que separa com uma reta
from sklearn.neural_network import MLPClassifier         # a rede neural (MLP)
from sklearn.inspection import DecisionBoundaryDisplay   # desenhar a região de decisão do modelo

## Etapa 2: gerar os dados

A célula abaixo cria **1000 visitantes fictícios**. Você **não precisa entender o código dela agora**. Só precisa saber que existe uma regra escondida que decide quem compra, e que o nosso modelo vai tentar **descobrir essa regra sozinho**, apenas olhando os exemplos.

In [ ]:
dados.to_csv('propensao_compra.csv')

Vamos olhar a tabela. `shape` mostra **(linhas, colunas)** e `head()` mostra as 5 primeiras linhas.

In [ ]:
print(dados.shape)
dados.head()

(1000, 3)


,tempo_no_site,desconto,comprou
0,15.5,1.2,0
1,8.8,9.2,1
2,17.2,2.6,0
3,13.9,3.0,0
4,1.9,12.6,0


In [ ]:
# proporção de quem comprou (1) e de quem não comprou (0)
dados['comprou'].value_counts(normalize=True).round(3)

Mais da metade dos visitantes **não compra**. Guarde essa informação: ela será a nossa régua para saber se o modelo é bom.

## Etapa 3: olhar os dados antes de modelar

Cada ponto é um visitante. **Vermelho = comprou**, **azul = não comprou**.

**Pergunta para a turma:** onde estão concentrados os compradores? Dá para separar vermelhos de azuis com **uma reta**?

In [ ]:
dados.plot.scatter(x='tempo_no_site', y='desconto', c='comprou', colormap='coolwarm',
                   figsize=(6, 5), title='Vermelho = comprou | Azul = não comprou');

## Etapa 4: separar entradas (X) e alvo (y), e depois treino e teste

Convenção do scikit-learn: **X** é a tabela com as colunas de entrada e **y** é a coluna que queremos prever.

### TODO 1
Crie `X` com as colunas `tempo_no_site` e `desconto` e `y` com a coluna `comprou`.

*Dica: para várias colunas use colchetes duplos, `dados[['a', 'b']]`. Para uma coluna, colchetes simples, `dados['a']`.*

In [ ]:
X = None  # TODO: tabela com as colunas tempo_no_site e desconto
y = None  # TODO: coluna comprou

In [ ]:
print('X tem', X.shape[0], 'linhas e', X.shape[1], 'colunas')
X.head()

Agora separamos **treino (75%)** e **teste (25%)**. É como estudar com uma lista de exercícios e ser avaliado em uma **prova com questões que você nunca viu**. Se o modelo fosse avaliado nos mesmos dados em que aprendeu, ele poderia só "decorar".

### TODO 2
Chame `train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)`. Ela devolve **4 coisas**, nesta ordem: `X_treino, X_teste, y_treino, y_teste`.

*(`random_state=42` repete o sorteio; `stratify=y` mantém a mesma proporção de compradores nos dois grupos.)*

In [ ]:
X_treino, X_teste, y_treino, y_teste = None, None, None, None  # TODO: chame train_test_split

In [ ]:
print('Treino:', len(X_treino), 'visitantes | Teste:', len(X_teste), 'visitantes')
print('Régua: se chutarmos "ninguém compra", acertamos', round(1 - y_teste.mean(), 3))

## Etapa 5: padronizar as colunas (StandardScaler)

Olhe as duas colunas: `tempo_no_site` e `desconto` têm escalas parecidas aqui, mas em dados reais isso quase nunca acontece (pense em `idade` e `salário`). Redes neurais **aprendem melhor quando todas as colunas estão na mesma escala**.

O `StandardScaler` transforma cada coluna para **média 0 e desvio padrão 1**. São duas regras:

- no **treino**: `fit_transform`, que **aprende** a média e o desvio e já converte
- no **teste**: só `transform`, que converte usando a média e o desvio **do treino** (o teste é "o futuro", não podemos espiar)

### TODO 3
Crie o `StandardScaler()` em `escala`, aplique `fit_transform` em `X_treino` e `transform` em `X_teste`.

In [ ]:
escala = None        # TODO: StandardScaler()
X_treino_esc = None  # TODO: escala.fit_transform(...) no treino
X_teste_esc = None   # TODO: escala.transform(...) no teste

Vamos conferir: antes e depois da padronização, as 3 primeiras linhas do treino. Depois, a média e o desvio de cada coluna padronizada (devem ser **0** e **1**). Um `-0.` significa um valor tão perto de zero que, arredondado, vira zero.

In [ ]:
print('Antes (valores originais):')
print(X_treino.head(3).values)
print('Depois (padronizados):')
print(X_treino_esc[:3].round(2))
print('Média de cada coluna:', X_treino_esc.mean(axis=0).round(2))
print('Desvio padrão de cada coluna:', X_treino_esc.std(axis=0).round(2))

## Etapa 6: primeiro modelo, uma reta

A **regressão logística** separa os dois grupos com **uma reta**. Todo modelo do scikit-learn segue o mesmo padrão de 3 passos:

1. **criar** o modelo
2. `.fit(...)`: **treinar** com os dados de treino (já padronizados)
3. `.score(...)`: **medir a acurácia** (a proporção de acertos) nos dados de teste (também padronizados)

In [ ]:
reta = LogisticRegression()
reta.fit(X_treino_esc, y_treino)
acc_reta = reta.score(X_teste_esc, y_teste)
print('Acurácia da reta:', round(acc_reta, 3))

Vamos **ver** o que a reta aprendeu. A cor do fundo é a **decisão do modelo** em cada região (vermelho = prevê compra, azul = prevê não compra). Os pontos são os visitantes do teste, com a cor **real**.

*Os eixos mostram os valores **padronizados** (0 = média, 1 = um desvio padrão acima da média). O desenho é o mesmo de antes, só mudou a régua.*

In [ ]:
tela = DecisionBoundaryDisplay.from_estimator(reta, X_teste_esc, response_method='predict', cmap='coolwarm', alpha=0.3)
tela.ax_.scatter(X_teste_esc[:, 0], X_teste_esc[:, 1], c=y_teste, cmap='coolwarm', edgecolor='k')
tela.ax_.set_xlabel('tempo_no_site (padronizado)')
tela.ax_.set_ylabel('desconto (padronizado)')
tela.ax_.set_title('A reta tenta separar quem compra de quem não compra');

**Leitura:** a acurácia da reta é parecida com a do chute "ninguém compra". Uma reta **não consegue desenhar uma faixa no meio**. Precisamos de um modelo que desenhe **curvas**.

## Etapa 7: segundo modelo, a rede neural (MLP)

A **MLP** (Multilayer Perceptron, perceptron de múltiplas camadas) é uma rede neural que consegue desenhar fronteiras curvas. Vamos construí-la em **3 passos**: montar a arquitetura, treinar e avaliar.

### 7.1 Montar a arquitetura

A arquitetura define **quantas camadas e quantos neurônios** a rede tem. A nossa terá **3 camadas**:

| Camada | Neurônios | O que faz |
|---|---|---|
| **Entrada** | 2 (`tempo_no_site` e `desconto`) | recebe os dados, não calcula nada |
| **Oculta** | 8 | cada neurônio soma as entradas multiplicadas por pesos e aplica a **função de ativação** |
| **Saída** | 1 | entrega a **probabilidade de comprar** (o scikit-learn monta esta camada sozinho) |

Cada argumento do `MLPClassifier` descreve uma parte da rede:

- `hidden_layer_sizes=(8,)`: **1 camada oculta com 8 neurônios** (`(8, 4)` seriam 2 camadas ocultas, com 8 e 4 neurônios)
- `activation='relu'`: a **função de ativação** dos neurônios ocultos. A ReLU devolve **0** se a soma for negativa e **a própria soma** se for positiva. É essa dobra que permite à rede desenhar curvas
- `max_iter=1000`: o máximo de **épocas** (voltas de treino sobre todos os exemplos)
- `random_state=42`: repete o sorteio dos **pesos iniciais**

### TODO 4
Crie a rede em uma variável chamada `rede`, com `hidden_layer_sizes=(8,)`, `activation='relu'`, `max_iter=1000` e `random_state=42`. **Ainda não treine.**

In [ ]:
rede = None  # TODO: MLPClassifier(hidden_layer_sizes=..., activation=..., max_iter=..., random_state=...)

In [ ]:
print(rede)

### 7.2 Treinar e medir

Criar a rede só monta a arquitetura, com pesos ainda sem sentido. O `.fit(...)` é o **treinamento**: a rede ajusta os pesos olhando os exemplos de treino. Depois, `.score(...)` mede a acurácia no teste.

### TODO 5
Treine a rede com `.fit(X_treino_esc, y_treino)` e guarde em `acc_mlp` o resultado de `.score(X_teste_esc, y_teste)`.

In [ ]:
# TODO: treinar com rede.fit(...)

acc_mlp = None  # TODO: rede.score(...)

In [ ]:
print('Acurácia da reta:', round(acc_reta, 3))
print('Acurácia da MLP: ', round(acc_mlp, 3))

### 7.3 Ver o que a rede aprendeu

In [ ]:
tela = DecisionBoundaryDisplay.from_estimator(rede, X_teste_esc, response_method='predict', cmap='coolwarm', alpha=0.3)
tela.ax_.scatter(X_teste_esc[:, 0], X_teste_esc[:, 1], c=y_teste, cmap='coolwarm', edgecolor='k')
tela.ax_.set_xlabel('tempo_no_site (padronizado)')
tela.ax_.set_ylabel('desconto (padronizado)')
tela.ax_.set_title('A rede neural desenha uma faixa curva');

**Leitura:** a rede **descobriu sozinha** a faixa do meio, onde os visitantes compram. Ninguém programou essa regra: ela foi aprendida dos exemplos.

Depois do treino, a rede guarda informações sobre a própria arquitetura. Vamos conferir se ela ficou como planejamos.

In [ ]:
print('Número de camadas (entrada + ocultas + saída):', rede.n_layers_)
print('Neurônios de saída:', rede.n_outputs_)
print('Ativação das camadas ocultas:', rede.activation)
print('Ativação da camada de saída:', rede.out_activation_)

A saída usa a ativação **`logistic`** (também chamada de **sigmoide**), que entrega um número entre 0 e 1: a **probabilidade de compra**.

## Etapa 8: usar o modelo em decisões de negócio

O modelo também devolve a **probabilidade** de compra com `.predict_proba(...)`. A coluna `[:, 1]` é a probabilidade de **comprar**.

Vamos testar 5 visitantes novos. Repare que os três primeiros ficaram **o mesmo tempo (10 min)** e só mudaram o **desconto**. Como a rede aprendeu com dados padronizados, **todo visitante novo precisa passar pela mesma padronização** (`escala.transform`).

In [ ]:
novos = pd.DataFrame({'tempo_no_site': [10, 10, 10, 1, 19],
                      'desconto':      [0, 10, 20, 10, 10]})
novos_esc = escala.transform(novos)    # mesma padronização do treino
novos

### TODO 6
Calcule `prob = rede.predict_proba(novos_esc)[:, 1]`.

In [ ]:
prob = None  # TODO: probabilidade de comprar de cada visitante novo

In [ ]:
novos['prob_compra'] = prob.round(3)
novos

**Perguntas de negócio:**

1. Para quem fica 10 minutos, o que acontece com a chance de compra quando o desconto sobe de 0% para 10% e para 20%? Vale a pena dar 20%?
2. E para quem fica só 1 minuto? O cupom resolve?
3. Quem fica 19 minutos também tem chance baixa. O que isso pode significar no comportamento real de um cliente? (Lembre: os dados aqui são inventados, mas a pergunta é real.)

## Etapa 9: experimentos, mude a arquitetura e a ativação

Volte à **Etapa 7**, mude **um argumento por vez** no `MLPClassifier`, rode de novo as células 7.1, 7.2 e 7.3 e anote a acurácia no teste.

**Experimento A: número de neurônios ocultos**

| `hidden_layer_sizes` | O que significa | Acurácia no teste |
|---|---|---|
| `(1,)` | 1 neurônio oculto | |
| `(2,)` | 2 neurônios ocultos | |
| `(3,)` | 3 neurônios ocultos | |
| `(8,)` | 8 neurônios ocultos | |

Agora fique com `(3,)` e troque `random_state=42` por **0**, depois por **1** e depois por **7**. O que acontece com a acurácia? O que isso sugere sobre o **sorteio inicial dos pesos**?

*Pense: a rede começa com pesos aleatórios. Com poucos neurônios, o resultado pode depender mais da sorte desse começo.*

**Experimento B: função de ativação** (com `hidden_layer_sizes=(8,)` e `random_state=42`)

| `activation` | O que é | Acurácia no teste |
|---|---|---|
| `'relu'` | 0 se negativo, o próprio valor se positivo | |
| `'tanh'` | curva em S que vai de -1 a 1 | |
| `'logistic'` | sigmoide, curva em S que vai de 0 a 1 | |
| `'identity'` | **nenhuma ativação** (a saída é a própria soma) | |

**Pergunta:** o que aconteceu com `'identity'`? Por que uma rede com 8 neurônios, mas **sem ativação**, se comporta como a reta da Etapa 6?

*Se aparecer um aviso `Maximum iterations (1000) reached`, a rede ainda estava aprendendo quando as épocas acabaram. Você pode aumentar `max_iter`.*

**Volte para `(8,)`, `'relu'` e `random_state=42` antes de seguir para a Etapa 10.**

## Etapa 10: espiando dentro da rede

Depois do treino, a rede guarda os **pesos** que aprendeu e o **erro** (perda) de cada época.

In [ ]:
print('Pesos entre as entradas e a camada oculta (entradas x neurônios):', rede.coefs_[0].shape)
print('Pesos entre a camada oculta e a saída (neurônios x saídas):', rede.coefs_[1].shape)
print('Épocas realizadas no treino:', rede.n_iter_)

In [ ]:
# o erro (perda) a cada época de treino
pd.Series(rede.loss_curve_).plot(title='O erro cai durante o treino', xlabel='época', ylabel='perda');

Contando os pesos e os valores de ajuste (viés): **(2×8) + 8 + (8×1) + 1 = 33 números** que a rede aprendeu sozinha.

## O que você usou hoje (e vamos explicar na teoria)

| Você viu no código | Termo da teoria |
|---|---|
| `tempo_no_site`, `desconto` | **entradas** (features) |
| `StandardScaler` | **padronização** das entradas |
| `hidden_layer_sizes=(8,)` | **camada oculta** e **neurônios** |
| `activation='relu'` | **função de ativação** |
| `rede.coefs_` | **pesos** |
| `rede.fit(...)` | **treinamento** (gradiente descendente e retropropagação) |
| `rede.loss_curve_` e `rede.n_iter_` | **perda** (erro) e **épocas** |
| `rede.predict_proba(...)` | saída da rede como **probabilidade** |
| `train_test_split` | **treino e teste** |

## Perguntas para guardar para a teoria

1. Por que uma reta não conseguiu resolver este problema e a MLP conseguiu?
2. O que é, de fato, um **neurônio** dentro dessa rede?
3. Por que a ativação `'identity'` transformou a rede em uma reta?
4. O que o `.fit` faz por baixo dos panos para os pesos mudarem?
5. Por que o erro cai a cada época?